# Regresja logistyczna jako pipeline
Ten sam model co w przykladowa regresja logistyczna, ale ladnie ulozony w pipeline

In [24]:
#importuje napisane przez nas funkcje i je aktualizuje na bieżąco, dzięki czemu nie trzeba restartować kernela przy każdej zmianie w funkcjach
%load_ext autoreload
%autoreload 2

from funkcje import *
from wskazniki_kredytowe import oblicz_wskazniki

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

## Transformer przygotowania danych



In [26]:

class PrzygotowanieCech(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        kolumny_do_pominiecia=None,
        prog_korelacji=0.9,
        prog_auc=0.6,
        tylko_wskazniki=False,
        prog_nan=0.5,
        prog_zer=0.5,
    ):
        self.kolumny_do_pominiecia = kolumny_do_pominiecia
        self.prog_korelacji = prog_korelacji
        self.prog_auc = prog_auc
        self.tylko_wskazniki = tylko_wskazniki
        self.prog_nan = prog_nan
        self.prog_zer = prog_zer

    def _przygotuj_baze(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Dane wejściowe muszą być obiektem pandas.DataFrame.")

        pomin = self.kolumny_do_pominiecia or []
        brakujace = sorted(set(pomin) - set(X.columns))
        if brakujace:
            raise ValueError(f"Brak wymaganych kolumn do pominięcia: {brakujace}")

        baza = X.drop(columns=pomin).copy()
        kolumny_poczatkowe = set(baza.columns)

        baza = oblicz_wskazniki(baza)

        if self.tylko_wskazniki:
            nowe_kolumny = [
                kolumna for kolumna in baza.columns
                if kolumna not in kolumny_poczatkowe
            ]
            baza = baza[nowe_kolumny]

        return baza

    @staticmethod
    def _kolumny_numeryczne(X):
        kolumny = X.select_dtypes(include=["float"]).columns.tolist()
        if "wsk_liczba_dni_istnienia" in X.columns and "wsk_liczba_dni_istnienia" not in kolumny:
            kolumny.append("wsk_liczba_dni_istnienia")
        return kolumny

    def _odrzuc_kolumny_niskiego_auc(self, X, y):
        rows = []
        for kolumna in X.columns:
            maska = X[kolumna].notna()
            if maska.sum() == 0 or y.loc[maska].nunique() < 2:
                continue
            auc = roc_auc_score(y.loc[maska], X.loc[maska, kolumna])
            rows.append((kolumna, max(auc, 1 - auc), auc >= 0.5))

        if not rows:
            raise ValueError("Nie udało się policzyć AUC dla żadnej cechy.")

        tabela = pd.DataFrame(rows, columns=["cecha", "auc", "zgodny_kierunek"])
        do_usuniecia = tabela.loc[tabela["auc"] < self.prog_auc, "cecha"].tolist()
        do_odwrocenia = tabela.loc[
            (tabela["auc"] >= self.prog_auc) & tabela["zgodny_kierunek"], "cecha"
        ].tolist()
        return do_usuniecia, do_odwrocenia

    def fit(self, X, y):
        y = pd.Series(y, index=X.index if isinstance(X, pd.DataFrame) else None)
        baza = self._przygotuj_baze(X)
        self.kolumny_numeryczne_ = self._kolumny_numeryczne(baza)
        if not self.kolumny_numeryczne_:
            raise ValueError("Nie znaleziono cech numerycznych.")

        numeryczne = baza[self.kolumny_numeryczne_].copy()
        numeryczne = numeryczne.replace([np.inf, -np.inf], np.nan)

        maska_nan = numeryczne.isna().mean() <= self.prog_nan
        maska_zer = numeryczne.eq(0).mean() <= self.prog_zer
        maska_jakosci = maska_nan & maska_zer

        self.kolumny_zbyt_wielu_brakow_ = numeryczne.columns[~maska_jakosci].tolist()
        numeryczne = numeryczne.loc[:, maska_jakosci]

        po_korelacji = usuwanie_najbardziej_scorrelowanych(
            numeryczne, prog=self.prog_korelacji
        )
        self.kolumny_po_korelacji_ = po_korelacji.columns.tolist()
        nie_monotoniczne = wykryj_niemonotoniczne(po_korelacji, y)
        self.kolumny_niemonotoniczne_ = list(nie_monotoniczne)

        if self.kolumny_niemonotoniczne_:
            po_zawiasach, self.progi_ = dodaj_cechy_zawiasowe(
                po_korelacji, y, self.kolumny_niemonotoniczne_
            )
        else:
            po_zawiasach, self.progi_ = po_korelacji, {}

        self.kolumny_do_usuniecia_, self.kolumny_do_odwrocenia_ = (
            self._odrzuc_kolumny_niskiego_auc(po_zawiasach, y)
        )
        wynik = po_zawiasach.drop(columns=self.kolumny_do_usuniecia_)
        wynik[self.kolumny_do_odwrocenia_] = -wynik[self.kolumny_do_odwrocenia_]

        skosnosc = wynik.skew()
        self.kolumny_do_log_ = [
            kolumna for kolumna in wynik.columns
            if skosnosc[kolumna] > 2 and wynik[kolumna].min() >= 0
        ]
        wynik[self.kolumny_do_log_] = np.log1p(wynik[self.kolumny_do_log_])
        self.kolumny_wynikowe_ = wynik.columns.tolist()
        return self

    def transform(self, X):
        baza = self._przygotuj_baze(X)
        numeryczne = baza.reindex(columns=self.kolumny_numeryczne_)
        numeryczne = numeryczne.replace([np.inf, -np.inf], np.nan)
        numeryczne = numeryczne.drop(columns=self.kolumny_zbyt_wielu_brakow_)
        numeryczne = numeryczne.reindex(columns=self.kolumny_po_korelacji_)

        if self.kolumny_niemonotoniczne_:
            numeryczne = dodaj_cechy_zawiasowe(
                numeryczne,
                kolumny_niemonotoniczne=self.kolumny_niemonotoniczne_,
                progi=self.progi_,
            )[0]

        numeryczne = numeryczne.drop(columns=self.kolumny_do_usuniecia_)
        numeryczne[self.kolumny_do_odwrocenia_] = -numeryczne[self.kolumny_do_odwrocenia_]
        numeryczne[self.kolumny_do_log_]=numeryczne[self.kolumny_do_log_].clip(lower=0)
        numeryczne[self.kolumny_do_log_] = np.log1p(numeryczne[self.kolumny_do_log_])
        return numeryczne.reindex(columns=self.kolumny_wynikowe_)


## Trenowanie i użycie pipeline'u

In [27]:
df = pd.read_csv("zbior_1.csv")
TARGET_COLUMN = df.columns[-1]
KOLUMNY_DO_POMINIECIA = [
    "szczegolnaFormaPrawna_Symbol",
    "schemat_wsk_rzis",
    "schemat_wsk_bilans",
]

X = df.drop(columns=TARGET_COLUMN)
y = df[TARGET_COLUMN]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=2137, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=2137, stratify=y_temp
)

pipeline = Pipeline([
    ("przygotowanie_cech", PrzygotowanieCech(KOLUMNY_DO_POMINIECIA)),
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        l1_ratio=1, solver="liblinear", C=100, random_state=2137, max_iter=1000
    )),
])

pipeline.fit(X_train, y_train)
prawdopodobienstwo = pipeline.predict_proba(X_valid)[:, 1]
predykcja = pipeline.predict(X_valid)

print(f"AUC: {roc_auc_score(y_valid, prawdopodobienstwo):.3f}")

AUC: 0.796


In [31]:
#optymalizacja parametrów pipeline'u
param_grid = {
    "przygotowanie_cech__prog_auc": [0.5],
    "przygotowanie_cech__tylko_wskazniki": [False],
    "przygotowanie_cech__prog_nan": [0.35, 0.5,0.65],
    "przygotowanie_cech__prog_zer": [0.3,0.4, 0.5],
    "classifier__C": [0.1,1,1.5],
}
wyniki = []
najlepszy_auc = -float("inf")
najlepszy_model = None
najlepsze_parametry = None

kombinacje = list(ParameterGrid(param_grid))

for i, parametry in enumerate(kombinacje, start=1):
    model = clone(pipeline)
    model.set_params(**parametry)

    try:
        model.fit(X_train, y_train)

        prawdopodobienstwo = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, prawdopodobienstwo)

        wyniki.append({**parametry, "auc": auc})

        if auc > najlepszy_auc:
            najlepszy_auc = auc
            najlepszy_model = model
            najlepsze_parametry = parametry

        print(f"[{i}/{len(kombinacje)}] AUC: {auc:.4f}")

    except ValueError as e:
        print(f"[{i}/{len(kombinacje)}] Błąd: {e}")

wyniki_df = (
    pd.DataFrame(wyniki)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)

print("\nNajlepsze parametry:")
print(najlepsze_parametry)
print(f"\nNajlepszy ROC AUC: {najlepszy_auc:.4f}")

display(wyniki_df)

[1/27] AUC: 0.8256
[2/27] AUC: 0.8275
[3/27] AUC: 0.8307
[4/27] AUC: 0.8251
[5/27] AUC: 0.8275
[6/27] AUC: 0.8307
[7/27] AUC: 0.8250
[8/27] AUC: 0.8275
[9/27] AUC: 0.8307
[10/27] AUC: 0.8338
[11/27] AUC: 0.8408
[12/27] AUC: 0.8373
[13/27] AUC: 0.8334
[14/27] AUC: 0.8427
[15/27] AUC: 0.8411
[16/27] AUC: 0.8337
[17/27] AUC: 0.8427
[18/27] AUC: 0.8411
[19/27] AUC: 0.8332
[20/27] AUC: 0.8374
[21/27] AUC: 0.8336
[22/27] AUC: 0.8318
[23/27] AUC: 0.8404
[24/27] AUC: 0.8377
[25/27] AUC: 0.8324
[26/27] AUC: 0.8404
[27/27] AUC: 0.8377

Najlepsze parametry:
{'classifier__C': 1, 'przygotowanie_cech__prog_auc': 0.5, 'przygotowanie_cech__prog_nan': 0.5, 'przygotowanie_cech__prog_zer': 0.4, 'przygotowanie_cech__tylko_wskazniki': False}

Najlepszy ROC AUC: 0.8427


,classifier__C,przygotowanie_cech__prog_auc,przygotowanie_cech__prog_nan,przygotowanie_cech__prog_zer,przygotowanie_cech__tylko_wskazniki,auc
0,1.0,0.5,0.50,0.4,False,0.842683
1,1.0,0.5,0.65,0.4,False,0.842683
2,1.0,0.5,0.65,0.5,False,0.841098
3,1.0,0.5,0.50,0.5,False,0.841098
4,1.0,0.5,0.35,0.4,False,0.840793
5,1.5,0.5,0.50,0.4,False,0.840366
6,1.5,0.5,0.65,0.4,False,0.840366
7,1.5,0.5,0.50,0.5,False,0.837744
8,1.5,0.5,0.65,0.5,False,0.837744
9,1.5,0.5,0.35,0.4,False,0.837439


In [38]:
#sprawdzenie najlpeszych pararmtrow na kroswalidacji
param_grid = {
    "przygotowanie_cech__prog_auc": [0.5],
    "przygotowanie_cech__tylko_wskazniki": [False],
    "przygotowanie_cech__prog_nan": [ 0.5,0.65],
    "przygotowanie_cech__prog_zer": [0.4, 0.5],
    "classifier__C": [1,2],
}
model = GridSearchCV(estimator=pipeline, param_grid=param_grid, scoring="roc_auc", cv=5)
model.fit(X_train, y_train)
model.best_params_, model.best_score_

({'classifier__C': 1,
  'przygotowanie_cech__prog_auc': 0.5,
  'przygotowanie_cech__prog_nan': 0.5,
  'przygotowanie_cech__prog_zer': 0.4,
  'przygotowanie_cech__tylko_wskazniki': False},
 np.float64(0.7458363705987318))